In [0]:
silver_txn = spark.table(
    "banking_catalog.banking_schema.silver_transactions"
)

In [0]:
from pyspark.sql.functions import min, max

silver_txn.select(
    min("timestamp").alias("min_date"),
    max("timestamp").alias("max_date")
).show(truncate=False)

+-------------------+-------------------+
|min_date           |max_date           |
+-------------------+-------------------+
|2022-09-01 00:00:00|2022-09-18 16:18:00|
+-------------------+-------------------+



In [0]:
from pyspark.sql.functions import *

silver_txn = spark.table(
    "banking_catalog.banking_schema.silver_transactions"
)

gold_daily_df = (
    silver_txn
    .groupBy(to_date("timestamp").alias("transaction_date"))
    .agg(
        count("*").alias("transaction_count"),
        sum("amount_paid").alias("total_amount"),
        sum(
            when(col("is_laundering") == 1, 1)
            .otherwise(0)
        ).alias("laundering_transaction_count"),
        sum(
            when(
                col("is_laundering") == 1,
                col("amount_paid")
            ).otherwise(0)
        ).alias("laundering_amount")
    )
)

In [0]:
display(gold_daily_df)

transaction_date,transaction_count,total_amount,laundering_transaction_count,laundering_amount
2022-09-01,1114864,6441442674807.30,322,151805387234.33
2022-09-03,207359,562282508312.07,391,2724331128.07
2022-09-18,11,61624.57,8,44392.84
2022-09-11,396,1086972325.75,232,977319096.50
2022-09-02,754397,5059368272410.93,408,259634178.28
2022-09-15,46,6033235.88,28,3964059.16
2022-09-17,23,301964.82,15,225527.48
2022-09-12,281,1266245112.24,170,364256685.67
2022-09-14,121,311225728.04,70,24098182.54
2022-09-07,482712,2263913197529.78,497,716529898.39


In [0]:
gold_daily_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.banking_schema.gold_daily_transactions"
    )

In [0]:
spark.sql("""
SELECT *
FROM banking_catalog.banking_schema.gold_daily_transactions
ORDER BY transaction_date
""").show()

+----------------+-----------------+----------------+----------------------------+-----------------+
|transaction_date|transaction_count|    total_amount|laundering_transaction_count|laundering_amount|
+----------------+-----------------+----------------+----------------------------+-----------------+
|      2022-09-01|          1114864|6441442674807.30|                         322|  151805387234.33|
|      2022-09-02|           754397|5059368272410.93|                         408|     259634178.28|
|      2022-09-03|           207359| 562282508312.07|                         391|    2724331128.07|
|      2022-09-04|           207421| 425080583857.98|                         407|    1804624349.21|
|      2022-09-05|           482602|1214661318709.47|                         471|    1354567350.42|
|      2022-09-06|           482056|2505941411667.22|                         531|   24612592261.08|
|      2022-09-07|           482712|2263913197529.78|                         497|     7165